# 01 – Ingest, clean, enrich (Part 1)
Run from the repo root with the three originals in `data/`.

## Task 1.1 – csv module only

In [ ]:
import csv, json
with open("data/appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    all_rows = list(reader)
print("Row count:", len(all_rows))

def parse_row(row):
    r = dict(row)
    r["age"] = int(r["age"])
    r["fee_inr"] = int(r["fee_inr"])
    r["prior_no_shows"] = int(r["prior_no_shows"])
    r["distance_km"] = float(r["distance_km"])
    return r

accepted, rejected = [], []
for row in all_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)
print("accepted:", len(accepted), "rejected:", len(rejected))
print("blank age:", sum(r["age"] == "" for r in rejected), "| blank distance:", sum(r["distance_km"] == "" for r in rejected))
with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

## Task 1.2 – pass 1 (starts again from the original file)

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("data/appointments.csv")
print(df.shape); df.info()
print(df.isnull().sum())
print("duplicates:", df.duplicated().sum())
print("raw channel spellings:", df["booking_channel"].nunique())

df = df.drop_duplicates();                                   print("(a) after drop duplicates:", len(df))
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("(b) channels:", sorted(df["booking_channel"].dropna().unique()), "| rows:", len(df))
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("(c) rows:", len(df), "| unreadable appt_date:", df["appt_date"].isna().sum())
df = df.dropna(subset=["appt_date", "status"]);              print("(d) after dropping missing date/status:", len(df))

## Task 1.3 – pass 2: validity, gaps, outliers

In [ ]:
bad_age = (df["age"] < 1) | (df["age"] > 100)
print("invalid ages:", bad_age.sum())
df.loc[bad_age, "age"] = np.nan
print("missing ages in total:", df["age"].isna().sum())
med_age, med_dist = df["age"].median(), df["distance_km"].median()
print("median age:", med_age, "| median distance:", med_dist)
df["age"] = df["age"].fillna(med_age).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(med_dist)

for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_capped = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: Q1={q1} Q3={q3} lower={lo} upper={hi} capped={n_capped}")
    df[col] = np.clip(df[col], lo, hi)

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("largest fee now:", df["fee_inr"].max())

## Task 1.4 – features and NumPy summary

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)
print("lead_days range:", df["lead_days"].min(), "-", df["lead_days"].max())
print("no-shows:", df["no_show"].sum(), "of", len(df), f"({round(100*df['no_show'].mean(),1)}%)")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("(a) column means:", X.mean(axis=0).round(2))
mask = X[:, 3] > 14
print("(b) lead_days > 14:", mask.sum(), f"({round(100*mask.mean(),1)}%)")
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("(c) |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

## Task 1.5 – enrich clinics from Open-Meteo

In [ ]:
import requests
clinics = pd.read_csv("data/clinics.csv")
clinics = clinics.rename(columns={"id": "clinic_id", "name": "clinic_name"})   # harmless if already named so
URL = "https://api.open-meteo.com/v1/forecast"

def get_temp(lat, lon, name):
    reason = "unknown"
    for attempt in (1, 2):
        try:
            r = requests.get(URL, params={"latitude": lat, "longitude": lon,
                                          "hourly": "temperature_2m", "forecast_days": 1}, timeout=10)
            if r.status_code != 200:
                reason = f"status {r.status_code}"; continue
            return r.json()["hourly"]["temperature_2m"][0]
        except (requests.exceptions.RequestException, KeyError, IndexError, ValueError) as e:
            reason = f"{type(e).__name__}: {e}"
    print(f"FAILED {name}: {reason}")
    return None

clinics["temp_c"] = [get_temp(r.latitude, r.longitude, r.clinic_name) for r in clinics.itertuples()]
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics)

## Task 1.6 – export clean table and SQL load script

In [ ]:
COLS = ["appointment_id","patient_id","clinic_id","booked_on","appt_date","age","gender","distance_km",
        "fee_inr","booking_channel","visit_type","reminder_sent","prior_no_shows","status","lead_days","no_show"]
out = df.sort_values("appointment_id").copy()
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")
out = out[COLS]
print("appointment_id unique:", out["appointment_id"].is_unique)
print("missing per column:\n", out.isnull().sum())
assert out["appointment_id"].is_unique and out.isnull().sum().sum() == 0
out.to_csv("appointments_clean.csv", index=False)
print(out.shape)

def sql_val(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "NULL"
    if isinstance(v, str):
        return f"'{v}'"
    return str(v)

n = 0
with open("load_data.sql", "w") as f:
    for r in clinics.to_dict("records"):                                   # clinics first (foreign key)
        vals = ",".join(sql_val(r[c]) for c in ["clinic_id","clinic_name","city","tier","latitude","longitude","temp_c"])
        f.write(f"INSERT INTO clinics VALUES ({vals});\n"); n += 1
    for r in out.to_dict("records"):
        vals = ",".join(sql_val(r[c]) for c in COLS)
        f.write(f"INSERT INTO appointments VALUES ({vals});\n"); n += 1
print("INSERT statements written:", n)

## Part 2 cross-check in pandas (Task 2.2)
Run after the SQL; compare with the query output.

In [ ]:
chk = out.merge(clinics[["clinic_id","clinic_name","city"]], on="clinic_id")
g = chk.groupby(["clinic_name","city"]).agg(appointments=("no_show","size"), no_shows=("no_show","sum"))
g["no_show_pct"] = (100 * g["no_shows"] / g["appointments"]).round(1)
print(g.sort_values("no_show_pct", ascending=False))